In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
pd.set_option('display.float_format', lambda x: '%.2f' % x)
np.set_printoptions(precision=10, suppress=True)

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

import optuna


# Carregar os Dados

In [2]:
df_clientes = pd.read_csv('datasets/dataset_clientes_pj.csv')

In [3]:
# Visualizar as colunas
df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   atividade_economica     500 non-null    object 
 1   faturamento_mensal      500 non-null    float64
 2   numero_de_funcionarios  500 non-null    int64  
 3   localizacao             500 non-null    object 
 4   idade                   500 non-null    int64  
 5   inovacao                500 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 23.6+ KB


In [4]:
# Visualizar os 10 primeiros registros
df_clientes.head(10)

,atividade_economica,faturamento_mensal,numero_de_funcionarios,localizacao,idade,inovacao
0,Comércio,713109.95,12,Rio de Janeiro,6,1
1,Comércio,790714.38,9,São Paulo,15,0
2,Comércio,1197239.33,17,São Paulo,4,9
3,Indústria,449185.78,15,São Paulo,6,0
4,Agronegócio,1006373.16,15,São Paulo,15,8
5,Serviços,1629562.41,16,Rio de Janeiro,11,4
6,Serviços,771179.95,13,Vitória,0,1
7,Serviços,707837.61,16,São Paulo,10,6
8,Comércio,888983.66,17,Belo Horizonte,10,1
9,Indústria,1098512.64,13,Rio de Janeiro,9,3


In [5]:
# Visualizar os 10 ultimos registros
df_clientes.tail(10)

,atividade_economica,faturamento_mensal,numero_de_funcionarios,localizacao,idade,inovacao
490,Indústria,215580.61,11,Belo Horizonte,7,3
491,Serviços,1050776.57,14,Vitória,8,0
492,Comércio,785671.05,15,Vitória,9,2
493,Serviços,658330.45,20,Belo Horizonte,4,8
494,Agronegócio,1643153.26,14,Rio de Janeiro,10,1
495,Serviços,1581841.42,17,Rio de Janeiro,8,2
496,Indústria,1291309.57,9,São Paulo,6,9
497,Serviços,2211489.85,10,Belo Horizonte,10,0
498,Agronegócio,1460860.46,12,Rio de Janeiro,5,3
499,Indústria,173684.43,13,Belo Horizonte,4,9


In [6]:
# Medidas estatisticas
df_clientes.describe()

,faturamento_mensal,numero_de_funcionarios,idade,inovacao
count,500.00,500.00,500.00,500.00
mean,1026715.63,13.69,9.25,4.39
std,420609.46,3.12,2.96,2.90
min,18421.22,2.00,0.00,0.00
25%,763253.58,12.00,7.00,2.00
50%,1022957.08,14.00,9.00,4.00
75%,1295888.52,16.00,11.00,7.00
max,2390677.22,21.00,16.00,9.00


# Preparar Dados para execução do modelo GMM

In [7]:
# Selecionar as colunas relevantes para a clusterização
X = df_clientes.copy()

# Separar variáveis por tipo para aplicar o ColumnTransformer
numeric_features = ['faturamento_mensal', 'numero_de_funcionarios', 'idade']
categorical_features = ['localizacao', 'atividade_economica']
ordinal_features = ['inovacao']

# Criar as transformações a serem aplicadas
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()
ordinal_transformer = OrdinalEncoder()

# Criar o ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('ord', ordinal_transformer, ordinal_features)
    ]
)

# Transformar os dados
X_transformed = preprocessor.fit_transform(X)

In [8]:
# Visualizar X_Transformed
X_transformed

array([[-0.7463449774, -0.5417919104, -1.1005884861, ...,  0.          ,
         0.          ,  1.          ],
       [-0.5616554761, -1.5035526981,  1.9434485069, ...,  0.          ,
         0.          ,  0.          ],
       [ 0.4058265391,  1.0611427358, -1.7770411512, ...,  0.          ,
         0.          ,  9.          ],
       ...,
       [ 2.8196246022, -1.1829657689,  0.2523168441, ...,  0.          ,
         1.          ,  0.          ],
       [ 1.0332141129, -0.5417919104, -1.4388148187, ...,  0.          ,
         0.          ,  3.          ],
       [-2.0301148644, -0.2212049812, -1.7770411512, ...,  1.          ,
         0.          ,  9.          ]])

# Treinar o modelo GMM

In [9]:
# Criar função para executar no Optuna
def gmm_objective(trial):
    # Definindo os hiperparâmetros a serem ajustados
    n_components = trial.suggest_int('n_components', 3, 10)
    covariance_type = trial.suggest_categorical('covariance_type', ['full', 'tied', 'diag', 'spherical'])

    # Instanciar o modelo GMM com os hiperparâmetros
    gmm = GaussianMixture(n_components=n_components, covariance_type=covariance_type, random_state=51)

    # Treinar o modelo nos dados
    gmm.fit(X_transformed)

    # Calculando o BIC (Bayesian Information Criteria)
    bic_gmm = gmm.bic(X_transformed)

    return bic_gmm

In [10]:
# Criar um estudo do Optuna
search_space = {'n_components': list(range(3, 11)),  # Número de componentes entre 3 e 10
                'covariance_type': ['full', 'tied', 'diag', 'spherical']}
sampler = optuna.samplers.GridSampler(search_space=search_space)
estudo_gmm = optuna.create_study(direction='maximize', sampler=sampler)

[I 2025-07-08 10:16:44,789] A new study created in memory with name: no-name-ab5f2314-8045-4d4d-af12-eb3b737955ee


In [11]:
# Executar o Optuna para otimizar os hiperparâmetros
estudo_gmm.optimize(gmm_objective, n_trials=32)

[I 2025-07-08 10:16:44,848] Trial 0 finished with value: -23479.73180905732 and parameters: {'n_components': 9, 'covariance_type': 'diag'}. Best is trial 0 with value: -23479.73180905732.
[I 2025-07-08 10:16:44,860] Trial 1 finished with value: -9599.639308249265 and parameters: {'n_components': 4, 'covariance_type': 'diag'}. Best is trial 1 with value: -9599.639308249265.


[I 2025-07-08 10:16:45,037] Trial 2 finished with value: -18618.230055933454 and parameters: {'n_components': 9, 'covariance_type': 'tied'}. Best is trial 1 with value: -9599.639308249265.
[I 2025-07-08 10:16:45,144] Trial 3 finished with value: -177.4763866904749 and parameters: {'n_components': 6, 'covariance_type': 'tied'}. Best is trial 3 with value: -177.4763866904749.
[I 2025-07-08 10:16:45,522] Trial 4 finished with value: -15471.558210902334 and parameters: {'n_components': 4, 'covariance_type': 'full'}. Best is trial 3 with value: -177.4763866904749.
[I 2025-07-08 10:16:45,547] Trial 5 finished with value: 13517.95476525887 and parameters: {'n_components': 3, 'covariance_type': 'spherical'}. Best is trial 5 with value: 13517.95476525887.
[I 2025-07-08 10:16:45,557] Trial 6 finished with value: 13234.27450643613 and parameters: {'n_components': 5, 'covariance_type': 'spherical'}. Best is trial 5 with value: 13517.95476525887.
[I 2025-07-08 10:16:45,593] Trial 7 finished with va

In [13]:
# Melhor configuração obtida pelo optuna
best_params = estudo_gmm.best_params

In [14]:
# Instanciar e treinar o modelo com melhores parâmetros do Optuna
best_gmm = GaussianMixture(n_components=best_params['n_components'], 
                           covariance_type=best_params['covariance_type'], 
                           random_state=51)

best_gmm.fit(X_transformed)

# Calcular o BIC do melhor modelo
best_bic = best_gmm.bic(X_transformed)

In [15]:
# Mostrar os melhores parâmetros e o BIC
print("Quantidade ideal de componentes:", best_params['n_components'])
print("Tipo de Covariância:", best_params['covariance_type'])
print("BIC do melhor modelo:", best_bic)

Quantidade ideal de componentes: 3
Tipo de Covariância: spherical
BIC do melhor modelo: 13517.95476525887
